In [1]:
#!pip install langchain langchain-core langchain-community pypdf pymupdf sentence-transformers chromadb

In [2]:
# sentence-transformer => to embedding the chukns
# vector database => chromadb
# langchain => create documents

In [1]:
from langchain_core.documents import Document

In [2]:
sample_doc = Document(
    page_content="Hello_word",
    metadata= {"source": "http://www.google.com"}
)

In [3]:
sample_doc

Document(metadata={'source': 'http://www.google.com'}, page_content='Hello_word')

In [4]:
from langchain_community.document_loaders import (
    TextLoader,
    PyPDFLoader
)

C:\Users\gaikw\AppData\Local\Temp\ipykernel_11424\3593821567.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import (


In [5]:
loader = TextLoader("data/python.txt", encoding="utf-8")

In [6]:
document = loader.load()

In [7]:
document

[Document(metadata={'source': 'data/python.txt'}, page_content='Python is a high-level, interpreted programming language that has become one of the most popular and widely used languages in the world. Created by Guido van Rossum and first released in 1991, Python emphasizes simplicity and readability, making it easy for beginners to learn while remaining powerful for experienced developers. Its clean and concise syntax allows programmers to write fewer lines of code compared to many other languages, enhancing productivity and maintainability. Python supports multiple programming paradigms, including procedural, object-oriented, and functional programming, which makes it versatile for a wide range of applications.\nSome key features and benefits of Python include:\n* Ease of Learning: Simple syntax and readability make Python beginner-friendly.\n* Versatility: Suitable for web development, data analysis, artificial intelligence, machine learning, scientific computing, automation, and mo

In [8]:
pdf_loader = PyPDFLoader("data/attension.pdf")

document = pdf_loader.load()


In [9]:
#document

# injesion Pipeline

In [10]:
import os


# documnent

In [11]:
def all_pdf_doc():
    folder_path = "data/pdfs"
    num_docs = 0
    all_docs = []

    for filename in os.listdir(folder_path):
        if filename.lower().endswith(".pdf"):

            pdf_path = os.path.join(folder_path, filename)

            loader = PyPDFLoader(pdf_path)
            doc = loader.load()
            all_docs.extend(doc)
            num_docs += 1

    print("total_pdf:", num_docs)
    print("total_pages:", len(all_docs))

    return all_docs

In [12]:
all_pdf_documents = all_pdf_doc()

total_pdf: 2
total_pages: 32


In [13]:
# chunks 
#!pip install langchain_text_splitters

#Chunks

In [14]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_doc(documents, chunk_size=500, chunk_overlap=50):

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap
    )

    chunked_docs = text_splitter.split_documents(documents)

    return chunked_docs

In [15]:
chunks = split_doc(all_pdf_documents)

In [16]:
len(chunks)

321

In [17]:
from sentence_transformers import SentenceTransformer

In [18]:
class EmbeddingManager:
    def __init__(self, model_name="all-MiniLM-L6-v2"):
        self.model_name = model_name

        print("loading model....", self.model_name)

        self.model = SentenceTransformer(self.model_name)

        print(
            "embedding dimensions=",
            self.model.get_sentence_embedding_dimension()
        )

    def generate_embeddings(self, text):
        embeddings = self.model.encode(text)

        print("embedding shape:", embeddings.shape)

        return embeddings

In [19]:
embedding_manager = EmbeddingManager()

loading model.... all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

embedding dimensions= 384


C:\Users\gaikw\AppData\Local\Temp\ipykernel_11424\1063170856.py:11: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  self.model.get_sentence_embedding_dimension()


In [20]:
test_text = ["This is a test sentence."]

result = embedding_manager.generate_embeddings(test_text)

print("FINAL SHAPE:", result.shape)

embedding shape: (1, 384)
FINAL SHAPE: (1, 384)


In [21]:
embedding_manager

In [22]:
import chromadb
import uuid

# Vector Store

In [23]:
class VectorStoreManager:
    def __init__(self, persist_directory = "data/vector_store", collection_name="pdf_documents"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.collection = None  # to store the embeddings and the documents(chunks) properly for symantic search
        self.client = None # to call the database, and to be stay connected with chromadb
        # Client नसता तर collection ला Python code मधून access/manage करणे शक्य झाले नसते.
        
        self._initialize_store()

    def _initialize_store(self):
        os.makedirs(self.persist_directory, exist_ok = True)

        #create a client
        self.client = chromadb.PersistentClient(path=self.persist_directory)

        #create the collection
        self.collection = self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={"description":"vector store collection for pdf embeddins in RAg"}
        )

        print("initialized the vector store with collection=", self.collection_name)
        print("docs in collecton ;", self.collection.count())

    def add_documents(self, documents, embeddings):  #documents => all chunks
        if len(documents) != len(embeddings):
            raise ValueError("num of documents does not match num of embeddings")


            #store => ids, embeddings, document, metadata in => COLLECTION

        ids = []
        all_metadata = []
        documents_content = []  # Atual Text(content) in Our Original Data
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc{uuid.uuid4()}"
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            all_metadata.append(metadata)
            
            documents_content.append(doc.page_content)
            
            embeddings_list.append(embedding.tolist())

            self.collection.add(
                ids=ids,
                metadatas=all_metadata,
                embeddings=embeddings_list
            )

        print("total doc added in vector store=", len(documents_content))
        print("docs in collection:", self.collection.count())

In [24]:
vector_store = VectorStoreManager()

initialized the vector store with collection= pdf_documents
docs in collecton ; 1284


In [25]:
texts = [doc.page_content for doc in chunks]

embedding = embedding_manager.generate_embeddings(texts)

vector_store.add_documents(chunks, embedding)

embedding shape: (321, 384)
total doc added in vector store= 321
docs in collection: 1605


In [26]:
from sklearn.metrics.pairwise import cosine_similarity

In [27]:
class RAGRetriever:
    def __init__(self, embedding_manager, vector_store):
        self.embedding_manager = embedding_manager
        self.vector_store = vector_store


    def retrieve(self, query, top_k=5, score_threshold=0.0):
        # query => embedding
        query_embeddings = self.embedding_manager.generate_embeddings([query])[0]

        # semantic search
        results = self.vector_store.collection.query(
            query_embeddings=[query_embeddings.tolist()],
            n_results=top_k
        )

        # cosine similarity
        retrieved_docs=[]
        
        if results["documents"] and results["documents"][0]:
            ids = results["ids"][0]
            metadatas = results["metadatas"][0]
            documents = results["documents"][0]
            distances = results["distances"][0]

            for i, (doc_id, metadata, document, distance) in enumerate(zip(ids, metadatas, documents, distances)):
                similarity_score = 1 - distance

                if similarity_score >= score_threshold:
                    retrieved_docs.append({
                        "id": doc_id,
                        "document": document,
                        "metadata": metadata,
                        "distance": distance,
                        "similarity_score": similarity_score,
                        "rank" : i + 1
                    })

            print(f"retrieved {len(retrieved_docs)} documents")

        else:
            print("no documents found")

        return retrieved_docs

In [28]:
rag_retriever = RAGRetriever(embedding_manager, vector_store)

In [29]:
rag_retriever.retrieve("What is encoder decoder")

embedding shape: (1, 384)
retrieved 5 documents


[{'id': 'docf0f910dc-976b-4b46-a3d5-f8ea7e0b7398',
  'document': None,
  'metadata': {'doc_index': 50,
   'publisher': 'Curran Associates, Inc.',
   'editors': 'I. Guyon and U.V. Luxburg and S. Bengio and H. Wallach and R. Fergus and S. Vishwanathan and R. Garnett',
   'page_label': '7',
   'created': '2017',
   'source': 'data/pdfs\\research1.pdf',
   'description': 'Paper accepted and presented at the Neural Information Processing Systems Conference (http://nips.cc/)',
   'published': '2017',
   'lastpage': '6008',
   'creationdate': '2026-09-21T05:17:28+00:00',
   'moddate': '2026-09-21T05:17:28+00:00',
   'firstpage': '5998',
   'title': 'Attention is All you Need',
   'producer': 'pdfcpu v0.12.1 dev',
   'type': 'Conference Proceedings',
   'creator': 'PyPDF',
   'author': 'Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Łukasz Kaiser, Illia Polosukhin',
   'subject': 'Neural Information Processing Systems http://nips.cc/',
   'page': 6,
  

In [ ]:
# OPENAI_API_KEY = ""

In [40]:
#!pip install langchain-openai

In [41]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    openai_api_key=OPENAI_API_KEY,
    model="gpt-6-astra",
    temperature=0.1,
    max_tokens=1024
)

In [56]:
# generate our retrieval-augmented output
def generate_output(query, retriever, llm, top_k=3):
    results = retriever.retrieve(query, top_k)

    context = "\n".join(
    str(doc.get("document"))
    for doc in results
        
    if doc.get("document") is not None
    ) if results else ""

    if not context:
        print("we found no relevant context for the given query")

    # context + query
    prompt = f""" use given context to generate the answer for the query
                Context: {context}
                Query: {query} """

    response = llm.invoke([prompt.format(context=context, query=query)]) # expecting a list as prompt
    return response.content

In [ ]:
answer = generate_output("what is RAG?", rag_retriever, llm)

In [ ]:
print(answer)

In [58]:
#!pip install langchain groq

In [ ]:
groq_api = ""

In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    groq_api_key=groq_api,
    model="qwen/qwen3-32b",
    temperature=0.1,
    max_tokens=1024
)

In [ ]:
# generate our retrieval-augmented output
def generate_output(query, retriever, llm, top_k=3):
    results = retriever.retrieve(query, top_k)

    context = "\n".join([doc["document"] for doc in results]) if results else ""

    if not context:
        print("we found no relevant context for the given query")

    # context + query
    prompt = f""" use given context to generate the answer for the query
                Context: {context}
                Query: {query} """

    response = llm.invoke([prompt.format(context=context, query=query)]) # expecting a list as prompt
    return response.content

In [ ]:
answer = generate_output("what is RAG?", rag_retriever, llm)

In [ ]:
print(answer)